# Tests for extractor

In [1]:
import os
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.append(str(ROOT))
from extractor.entities_extractor import EntitiesExtractor
from extractor.relations_extractor import RelationsExtractor
from extractor import Extractor
from config.llm import llm, embeddings

## extract entities

In [2]:
text = """
The information stored in memory provides historical retrieval and reasoning information for reflec-
tion. After a two-step exploration, we dynamically update the searched subgraph GSub, reasoning
paths P, and sub-objective status S in memory based on the ongoing reasoning process.
"""

entity_extractor = EntitiesExtractor(llm=llm, embeddings=embeddings)
entities = entity_extractor.extract_entities(text)

In [ ]:
bad_text = """
ksd sdm
"""

entity_extractor = EntitiesExtractor(llm=llm, embeddings=embeddings)
entities = entity_extractor.extract_entities(bad_text)

No entities found in the text.


## extract relations

In [3]:
text = """
The information stored in memory provides historical retrieval and reasoning information for reflec-
tion. After a two-step exploration, we dynamically update the searched subgraph GSub, reasoning
paths P, and sub-objective status S in memory based on the ongoing reasoning process.
"""

relation_extractor = RelationsExtractor(llm=llm, embeddings=embeddings)
relations = relation_extractor.extract_relations(text)

In [2]:
bad_text = """
4.1
Experimental Setups
"""

relation_extractor = RelationsExtractor(llm=llm, embeddings=embeddings)
relations = relation_extractor.extract_entities(bad_text)

## extract both entities and relations

In [6]:
text = """
The information stored in memory provides historical retrieval and reasoning information for reflec-
tion. After a two-step exploration, we dynamically update the searched subgraph GSub, reasoning
paths P, and sub-objective status S in memory based on the ongoing reasoning process.
"""

extractor = Extractor(llm=llm, embeddings=embeddings)
relations, entities = extractor.extract_relations_and_entities(text)

Calling llm...
Embedding relations and entities...


In [2]:
bad_text = """
PREFIX ns: <\protect\vrule width0pt\protect\href{http://rdf.freebase.com/ns/}{http
://rdf.freebase.com/ns/}>
SELECT DISTINCT ?tailEntity
WHERE {
"""

extractor = Extractor(llm=llm, embeddings=embeddings)
relations, entities = extractor.extract_relations_and_entities(bad_text)

No relations found in the text.


TypeError: cannot unpack non-iterable NoneType object

In [2]:
long_text = """
Memory Updating
The information stored in memory provides historical retrieval and reasoning information for reflec-
tion. After a two-step exploration, we dynamically update the searched subgraph GSub, reasoning
paths P, and sub-objective status S in memory based on the ongoing reasoning process.
Subgraph. The subgraph includes all retrieved relations and entities from the KG. We update the
subgraph in memory, which can be utilized during later reflection to determine which entity to
backtrack to for self-correction. In the D-th iteration, the searched subgraph GSub is updated by
adding the retrieved candidate relation set RD
cand and candidate entity set ED
cand.
Reasoning Paths. In order to ensure that the LLM can understand relationships between entities for
better reasoning and allow for path correction in reflection stage, we update reasoning paths P to
preserve the semantic structure within the KG.
Sub-Objective Status. The LLM may forget partial conditions in the reasoning process. Sub-
objectives obtained by decomposing the question can help the LLM remember multiple conditions
in the question. The status of sub-objectives contains the current known information related to the
sub-objectives, which can aid the LLM in remembering the known information of each condition and
determining whether to correct the exploration direction in reflection stage. Hence, we leverage the
LLM to update the currently known information relevant to sub-objectives into sub-objective status
S = {s1, s2, s3, ...}, |S| = |O|, based on the semantic information of the question q, sub-objectives
O, historical sub-objective status, and reasoning paths P, along with the LLM’s own knowledge. The
prompt is shown in Appendix A.3.
After the path exploration and memory updating, PoG prompts the LLM to reason whether the current
acquired information, including sub-objective states and reasoning paths recorded in memory, is
sufficient to infer an answer. The prompt is shown in Appendix A.4.1. If the LLM determines that the
information is sufficient, it will integrate reasoning paths, sub-objective states, and its own knowledge
to provide an answer. When information is considered insufficient, there may be two situations.
One is that PoG will acquire sufficient information after further extension of current paths, and the
other is that current paths are incorrect. Since the reasoning capability of the LLM does not always
guarantee the correctness of path exploration, there is a need to self-correct erroneous reasoning
paths. Therefore, we design a reflection mechanism to determine whether and how to self-correct
reasoning paths. When the LLM believes that the information is insufficient, PoG enters the stage of
reflection. Specifically, PoG utilizes the LLM to reflect on whether to correct the current exploration
direction based on the question q, sub-objective status S, reasoning paths P, and entities planned
for the next iteration of retrieval ED from memory. Besides, the LLM will provide the reason for
the reflection result. If the LLM judges that it is necessary to incorporate additional entities beyond
those in ED for exploration, then a self-correction of reasoning paths is needed. Otherwise, PoG will
continue exploring along the current reasoning paths with tail entities in ED. For self-correction,
PoG employs the LLM to decide which entities in Ecand = E1
cand ∪E2
cand ∪... ∪ED
cand to backtrack to
based on sub-objective states in S and the reason for additional retrieval obtained from the reflection,
and adds new exploration of backtracked entities ED
add into ED for the self-correction, denoted as
ED = ED ∪ED
add. The prompts for reflection are shown in Appendix A.4.2.
"""

extractor = Extractor(llm=llm, embeddings=embeddings)
relations, entities = extractor.extract_relations_and_entities(long_text)

Calling llm...
Embedding relations and entities...


In [2]:
text1 = """
• We propose a novel self-correcting adaptive planning paradigm for KG-augmented LLM
named PoG, which exploits the LLM to plan the adaptive breadth of reasoning paths and
reflect to self-correct erroneous paths. To the best of our knowledge, we are the first to
incorporate a reflection mechanism for self-correction and adaptive KG exploration into
KG-augmented LLMs, effectively augmenting the LLM’s reasoning ability.
"""

text2 = """
• We specially design Guidance, Memory, and Reflection mechanisms for PoG. Guidance
harnesses question conditions to better plan adaptive exploration by decomposing task into
sub-objectives including conditions. Memory records the subgraph, reasoning paths, and
sub-objective status to provide historical retrieval and reasoning information for Reflection.
Based on Memory, Reflection reasons whether to self-correct reasoning paths and which entity
to backtrack to for initiating new exploration.
"""

texts = [text1, text2]
extractor = Extractor(llm=llm, embeddings=embeddings)

for text in texts:
    relations, entities = extractor.extract_relations_and_entities(text)